### Atlas export - data for the UMAP scatter and the microbe card

Produces the static files the `/atlas` page loads. Nothing here runs at request
time; the page does array lookups.

| file | contents |
|---|---|
| `umap.f32.bin` | float32 `[n, 2]`, UMAP of the SNE matrix - the scatter |
| `nbr_sne_idx.i16.bin` / `nbr_sne_sim.f32.bin` | each OTU's K nearest **ecological** neighbours |
| `nbr_phylo_idx.i16.bin` / `nbr_phylo_sim.f32.bin` | the same from PhyloE - the **phylogenetic** column of the card |
| `otus.json` | one record per OTU: taxonomy, genome linkage, traits, neighbour overlap |
| `meta.json` | array shapes and dtypes, and the colour-by levels for the legend |

Row `i` is the same OTU in every one of these: `otus.json[i]`, points
`umap[2i]`/`umap[2i+1]`, neighbour block `[i*K : (i+1)*K]`. Neighbour indices
are row numbers into that same order, so the card resolves a neighbour without
a lookup table.

`nbr_overlap` in `otus.json` is how many of an OTU's K ecological neighbours are
also among its K phylogenetic ones. It is the per-OTU version of Fig. 4 - for
most OTUs it is close to zero, which is the point the card is making.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import umap
from sklearn.neighbors import NearestNeighbors

In [2]:
# ---- inputs (same layout as traits_predict.ipynb) ------------------------- #
SNE_FILE    = "social_niche_embedding_100.txt"
PHYLO_FILE  = "phylo_embed_PCA_100.txt"      # PhyloE, Methods L457-463
TAXMAP      = "taxmap_slv_ssu_ref_nr_138.2.txt"
TRAITOR_CSV = "trait_predcit.csv"            # its index = the genome-linked OTUs
TRAIT_TABLE = "../data/traits_predict/trait_table.csv"   # from traits_predict.ipynb

OUT = Path("../data/web")
OUT.mkdir(parents=True, exist_ok=True)

K = 50          # neighbours kept per OTU; halving K halves the neighbour files

In [3]:
def load_embedding(path):
    emb = pd.read_csv(path, header=None, sep=" ", low_memory=False, index_col=0)
    return emb.drop(index="<unk>", errors="ignore")


def load_taxonomy():
    """SILVA ranks indexed by "<accession>.<start>.<stop>", i.e. by OTU id."""
    tax = pd.read_csv(TAXMAP, sep="\t", low_memory=False)
    ranks = tax["path"].str.split(";", expand=True).iloc[:, :7]
    ranks.columns = ["kingdom", "phylum", "class", "order", "family", "genus", "species"]
    ranks.index = (tax.iloc[:, 0].astype(str) + "." +
                   tax.iloc[:, 1].astype(str) + "." +
                   tax.iloc[:, 2].astype(str)).values
    return ranks


def top_neighbours(mat, k):
    """Row numbers and cosine similarity of each row's k nearest *other* rows."""
    nn = NearestNeighbors(n_neighbors=k + 1, metric="cosine").fit(mat)
    dist, idx = nn.kneighbors(mat)
    sim = 1.0 - dist

    out_idx = np.empty((len(mat), k), np.int16)
    out_sim = np.empty((len(mat), k), np.float32)
    for r in range(len(mat)):
        keep = idx[r] != r                       # an OTU is not its own neighbour
        out_idx[r] = idx[r][keep][:k]
        out_sim[r] = sim[r][keep][:k]
    return out_idx, out_sim


def column(series, ndigits=None):
    """Python list with NaN as None - json.dump writes bare NaN, which is not JSON."""
    values = series.tolist()
    if ndigits is not None:
        return [None if v is None or v != v else round(float(v), ndigits) for v in values]
    return [None if isinstance(v, float) and v != v else v for v in values]

## Load

The SNE table sets the row order. PhyloE has to cover the same OTUs, since both
neighbour files are indexed by that one order; any OTU missing from PhyloE is
dropped from the export rather than left with an empty phylogenetic column.

In [5]:
sne = load_embedding(SNE_FILE)
phylo = load_embedding(PHYLO_FILE)

ids = sne.index[sne.index.isin(phylo.index)]
print(f"{len(ids)} OTUs exported; dropped {len(sne) - len(ids)} missing from PhyloE")
assert len(ids) < 2 ** 15, "more than 32767 OTUs - neighbour indices need int32"

sne = sne.loc[ids]
phylo = phylo.loc[ids]

tax = load_taxonomy().reindex(ids)
traits = pd.read_csv(TRAIT_TABLE, index_col="otu_id").reindex(ids)
genome_linked = ids.isin(pd.read_csv(TRAITOR_CSV, index_col=0).index)

# every column that has a matching "<name>_source" is a trait
TRAIT_NAMES = [c for c in traits.columns if f"{c}_source" in traits.columns]
print(f"{len(TRAIT_NAMES)} traits, {genome_linked.sum()} genome-linked OTUs")

14093 OTUs exported; dropped 0 missing from PhyloE
12 traits, 1112 genome-linked OTUs


## UMAP and neighbours

Cosine throughout: the SNE vectors are compared by direction everywhere else in
the paper, and PhyloE is a PCA of a distance matrix, so neither is meaningful
under Euclidean scale.

In [6]:
coords = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine",
                   random_state=0).fit_transform(sne.values).astype(np.float32)

nbr_sne_idx, nbr_sne_sim = top_neighbours(sne.values, K)
nbr_phylo_idx, nbr_phylo_sim = top_neighbours(phylo.values, K)

# how much of the ecological neighbourhood is also the phylogenetic one
overlap = np.array([len(set(a) & set(b))
                    for a, b in zip(nbr_sne_idx, nbr_phylo_idx)])
print(f"neighbour overlap out of {K}: mean {overlap.mean():.1f}, "
      f"median {np.median(overlap):.0f}, zero for {(overlap == 0).mean():.0%} of OTUs")

/home/dongbiao/.conda/envs/jupyter_notebook/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/dongbiao/.conda/envs/jupyter_notebook/lib/python3.9/site-packages/umap/umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


neighbour overlap out of 50: mean 2.2, median 1, zero for 31% of OTUs


## Write

In [7]:
for name, arr in [("umap.f32.bin", coords),
                  ("nbr_sne_idx.i16.bin", nbr_sne_idx),
                  ("nbr_sne_sim.f32.bin", nbr_sne_sim),
                  ("nbr_phylo_idx.i16.bin", nbr_phylo_idx),
                  ("nbr_phylo_sim.f32.bin", nbr_phylo_sim)]:
    (OUT / name).write_bytes(arr.tobytes())

In [8]:
tax_records = tax.astype(object).where(tax.notna(), None).to_dict("records")
trait_columns = {t: (column(traits[t]), column(traits[f"{t}_source"]),
                     column(traits[f"{t}_prob"], 3), column(traits[f"{t}_auc"], 3))
                 for t in TRAIT_NAMES}

otus = [
    {"i": i, "id": otu, **tax_records[i],
     "genome_linked": bool(genome_linked[i]),
     "nbr_overlap": int(overlap[i]),
     "traits": {t: {"value": value[i], "source": source[i],
                    "prob": prob[i], "auc": auc[i]}
                for t, (value, source, prob, auc) in trait_columns.items()}}
    for i, otu in enumerate(ids)
]
(OUT / "otus.json").write_text(json.dumps(otus, separators=(",", ":")))

16109738

In [9]:
def levels(series):
    """Distinct values, commonest first - the legend order for a colour-by field."""
    return column(pd.Series(series.value_counts().index))


meta = {
    "n_otus": len(ids),
    "k_neighbours": K,
    "arrays": {
        "umap.f32.bin":           {"dtype": "float32", "shape": [len(ids), 2]},
        "nbr_sne_idx.i16.bin":    {"dtype": "int16",   "shape": [len(ids), K]},
        "nbr_sne_sim.f32.bin":    {"dtype": "float32", "shape": [len(ids), K]},
        "nbr_phylo_idx.i16.bin":  {"dtype": "int16",   "shape": [len(ids), K]},
        "nbr_phylo_sim.f32.bin":  {"dtype": "float32", "shape": [len(ids), K]},
    },
    "layout": "row-major; row i of every array is otus.json[i]; "
              "neighbour values are row numbers into that same order",
    "color_by": {"phylum": levels(tax["phylum"]),
                 **{t: levels(traits[t]) for t in TRAIT_NAMES}},
    "inputs": {"sne": SNE_FILE, "phylo": PHYLO_FILE,
               "taxonomy": TAXMAP, "traits": TRAIT_TABLE},
}
(OUT / "meta.json").write_text(json.dumps(meta, indent=1))

for f in sorted(OUT.iterdir()):
    print(f"{f.name:24s} {f.stat().st_size / 1e6:7.2f} MB")

meta.json                   0.00 MB
nbr_phylo_idx.i16.bin       1.41 MB
nbr_phylo_sim.f32.bin       2.82 MB
nbr_sne_idx.i16.bin         1.41 MB
nbr_sne_sim.f32.bin         2.82 MB
otus.json                  16.11 MB
umap.f32.bin                0.11 MB


## Sanity checks

In [10]:
n = len(ids)

# the binaries are exactly the declared shape, and reload to what we wrote
for name, spec in meta["arrays"].items():
    got = np.fromfile(OUT / name, dtype=spec["dtype"])
    assert got.size == spec["shape"][0] * spec["shape"][1], name
assert np.array_equal(np.fromfile(OUT / "umap.f32.bin", "float32").reshape(-1, 2), coords)

# neighbours are in range, never self, never repeated within a row
for idx in (nbr_sne_idx, nbr_phylo_idx):
    assert idx.min() >= 0 and idx.max() < n
    assert (idx != np.arange(n)[:, None]).all()
    assert all(len(set(row)) == K for row in idx)

# similarities are sorted best-first and agree with a directly computed cosine
assert (np.diff(nbr_sne_sim, axis=1) <= 1e-6).all()
row = 0
v = sne.values[row] / np.linalg.norm(sne.values[row])
other = sne.values[nbr_sne_idx[row, 0]]
assert abs(v @ (other / np.linalg.norm(other)) - nbr_sne_sim[row, 0]) < 1e-5

# otus.json lines up with the arrays
loaded = json.loads((OUT / "otus.json").read_text())
assert len(loaded) == n
assert [r["id"] for r in loaded] == list(ids)
assert all(r["i"] == i for i, r in enumerate(loaded))
assert set(loaded[0]["traits"]) == set(TRAIT_NAMES)

print("all checks passed")
loaded[0]

all checks passed


{'i': 0,
 'id': 'AAAA02020714.1.1202',
 'kingdom': 'Bacteria',
 'phylum': 'Pseudomonadota',
 'class': 'Alphaproteobacteria',
 'order': 'Sphingomonadales',
 'family': 'Sphingomonadaceae',
 'genus': 'Sphingomonas',
 'species': '',
 'genome_linked': False,
 'nbr_overlap': 1,
 'traits': {'Oxygen_Preference': {'value': 'aerobic',
   'source': 'SNE_predicted',
   'prob': 0.538,
   'auc': 0.869},
  'Gram_Status': {'value': 'negative',
   'source': 'SNE_predicted',
   'prob': 0.607,
   'auc': 0.791},
  'Motility': {'value': 0,
   'source': 'SNE_predicted',
   'prob': 0.721,
   'auc': 0.639},
  'Spore_Formation': {'value': 0,
   'source': 'SNE_predicted',
   'prob': 0.925,
   'auc': 0.564},
  'Lactose': {'value': 0,
   'source': 'SNE_predicted',
   'prob': 0.721,
   'auc': 0.689},
  'Salicin': {'value': 0,
   'source': 'SNE_predicted',
   'prob': 0.709,
   'auc': 0.628},
  'Glycerol': {'value': 0,
   'source': 'SNE_predicted',
   'prob': 0.822,
   'auc': 0.609},
  'Melibiose': {'value': 0,
   '